# DSFB Semiconductor SECOM Benchmark

This notebook runs the `dsfb-semiconductor` Rust crate against the real SECOM semiconductor manufacturing dataset and writes a timestamped artifact bundle under `output-dsfb-semiconductor/<timestamp>_dsfb-semiconductor_<dataset>/`.

What this notebook does:

- obtains or reuses the real SECOM dataset,
- runs the deterministic DSFB residual / drift / slew / grammar / heuristics-bank pipeline,
- inspects the bundle manifest, archive-layout note, lead-time summary, density summary, nuisance proxies, and DRSC artifact paths,
- renders the generated figures inline, including the DRSC figure when present,
- exposes the generated PDF report and ZIP bundle for download.

What this notebook does **not** claim:

- no SEMI standards-compliance claim,
- no production-readiness claim,
- no universal-superiority claim over SPC/EWMA/FDC/ML baselines,
- no PHM 2018 support claim beyond the manual-placement contract and archive probe unless that real archive is present,
- no claim that the calibration workflow proves universal early-warning superiority.

This notebook is a real-data SECOM path first. PHM 2018 remains a stronger long-horizon target, but it is not claimed as implemented here unless the archive is manually supplied and verified.


In [ ]:
from pathlib import Path
import os

REPO = Path('/content/dsfb')
if not REPO.exists():
    !git clone https://github.com/infinityabundance/dsfb.git {REPO}
%cd /content/dsfb
print('Repo root:', REPO)

In [ ]:
import os
import shutil

if shutil.which('cargo') is None:
    !curl https://sh.rustup.rs -sSf | sh -s -- -y
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ.get('PATH', '')}"

if shutil.which('pdflatex') is None:
    !apt-get update -y
    !apt-get install -y texlive-latex-base pkg-config libfreetype6-dev libfontconfig1-dev

!cargo --version
!pdflatex --version | head -n 1

In [ ]:
import json
import subprocess
from pathlib import Path

command = [
    'cargo',
    'run',
    '--manifest-path',
    'crates/dsfb-semiconductor/Cargo.toml',
    '--',
    'run-secom',
    '--fetch-if-missing',
]
result = subprocess.run(command, check=True, text=True, capture_output=True)
print(result.stdout)
if result.stderr.strip():
    print(result.stderr)

run_dir = None
for line in result.stdout.splitlines():
    if line.startswith('Run directory: '):
        run_dir = Path(line.split(': ', 1)[1].strip())
        break
if run_dir is None:
    raise RuntimeError('Could not parse run directory from CLI output')

print('Parsed run directory:', run_dir)
assert run_dir.exists()

In [ ]:
import json
from pathlib import Path

metrics = json.loads((run_dir / 'benchmark_metrics.json').read_text())
archive_layout = json.loads((run_dir / 'secom_archive_layout.json').read_text())
baseline_summary = json.loads((run_dir / 'baseline_comparison_summary.json').read_text())
artifact_manifest = json.loads((run_dir / 'artifact_manifest.json').read_text())
phm_status = json.loads((run_dir / 'phm2018_support_status.json').read_text())

summary = metrics['summary']
print('Dataset summary:')
print(json.dumps(summary['dataset_summary'], indent=2))
print('\nTop-level benchmark counts:')
for key in [
    'analyzable_feature_count',
    'threshold_alarm_points',
    'ewma_alarm_points',
    'dsfb_raw_boundary_points',
    'dsfb_persistent_boundary_points',
    'dsfb_raw_violation_points',
    'dsfb_persistent_violation_points',
    'failure_runs',
    'failure_runs_with_preceding_dsfb_persistent_boundary_signal',
    'failure_runs_with_preceding_dsfb_persistent_violation_signal',
    'failure_runs_with_preceding_ewma_signal',
    'failure_runs_with_preceding_threshold_signal',
]:
    print(f'  {key}: {summary[key]}')

print('\nArtifact manifest:')
print(json.dumps(artifact_manifest, indent=2))
print('\nArchive-layout note:')
print(json.dumps(archive_layout, indent=2))

print('\nLead-time, density, and nuisance summary:')
print(json.dumps({
    'lead_time_summary': baseline_summary['lead_time_summary'],
    'density_summary': baseline_summary['density_summary'],
    'boundary_episode_summary': baseline_summary['boundary_episode_summary'],
    'pass_run_nuisance_proxy': baseline_summary['pass_run_nuisance_proxy'],
}, indent=2))

print('\nPHM 2018 support status:')
print(json.dumps(phm_status, indent=2))


In [ ]:
# Optional bounded calibration run. Uncomment to execute.
# import subprocess
# calibration_command = [
#     'cargo',
#     'run',
#     '--manifest-path',
#     'crates/dsfb-semiconductor/Cargo.toml',
#     '--',
#     'calibrate-secom',
#     '--healthy-pass-runs-grid', '80,100,120',
#     '--drift-window-grid', '3,5',
#     '--boundary-fraction-of-rho-grid', '0.4,0.5',
#     '--state-confirmation-steps-grid', '1,2',
#     '--persistent-state-steps-grid', '1,2',
#     '--density-window-grid', '10',
#     '--pre-failure-lookback-runs-grid', '10,20',
# ]
# calibration = subprocess.run(calibration_command, check=True, text=True, capture_output=True)
# print(calibration.stdout)


In [ ]:
from IPython.display import Image, display, Markdown

figure_names = sorted(path.name for path in (run_dir / 'figures').glob('*.png'))
for name in figure_names:
    figure_path = run_dir / 'figures' / name
    display(Markdown(f'## {name}'))
    display(Image(filename=str(figure_path)))

In [ ]:
from pathlib import Path

pdf_path = run_dir / 'engineering_report.pdf'
zip_path = run_dir / 'run_bundle.zip'

print('PDF:', pdf_path)
print('ZIP:', zip_path)

try:
    from google.colab import files
    if pdf_path.exists():
        files.download(str(pdf_path))
    files.download(str(zip_path))
except Exception as err:
    print('Colab download helper not available in this runtime:', err)
    print('Download the files directly from the paths above.')